In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# read csv files
discharge = pd.read_csv('CZO_2021.csv', parse_dates=['Date_Time'], index='Date_Time')
depth = pd.read_csv('depth.csv', parse_dates=['datetime'], index='datetime')

# replace these with your actual column names
discharge = discharge.rename(columns={'LowerLaJara': 'discharge_cfs'})
depth = depth.rename(columns={'P3': 'depth_m'})

TypeError: read_csv() got an unexpected keyword argument 'index'

Join the data on matching date times 

In [ ]:
# keep only timestamps that exist in BOTH datasets
rating_df = discharge.join(depth, how='inner')
# remove missing values
rating_df = rating_df.dropna()

print(rating_df.head())
print(rating_df.describe())

# plot discharge vs depth
plt.figure(figsize=(6,5))
plt.scatter(rating_df['discharge_cfs'], rating_df['depth_m'], alpha=0.6)
plt.xlabel('Discharge (cfs)')
plt.ylabel('Depth (m)')
plt.title('CZO discharge vs stilling well depth')

plt.grid(alpha=0.3)
plt.show()

Fit a linear rating curve

In [ ]:
# predictor variable must be 2D
X = rating_df[['discharge_cfs']]
y = rating_df['depth_m']

# fit regression
model = LinearRegression()
model.fit(X, y)
# predicted depths
y_pred = model.predict(X)

r2 = r2_score(y, y_pred)
# regression coefficients
slope = model.coef_[0]
intercept = model.intercept_

print(f"Slope: {slope:.5f}")
print(f"Intercept: {intercept:.5f}")
print(f"R²: {r2:.4f}")

Plot best line fit

In [ ]:
plt.figure(figsize=(6,5))

plt.scatter(rating_df['discharge_cfs'], rating_df['depth_m'], alpha=0.5, label='Observed data') # scatter points
plt.plot(rating_df['discharge_cfs'], y_pred, linewidth=2, label=f'Fit: depth = {slope:.4f}Q + {intercept:.4f}\nR² = {r2:.3f}')
plt.xlabel('Discharge (cfs)')
plt.ylabel('Depth (m)')
plt.title('CZO discharge to La Jara depth rating curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Convert discharge to depth

In [ ]:
full_df = discharge.copy()

# calculate predicted depth from discharge
full_df['predicted_depth_m'] = (
    slope * full_df['discharge_cfs'] + intercept
)
# join observed depths onto full dataframe
full_df = full_df.join(depth[['depth_m']], how='left')
# use observed depth where available, otherwise use predicted depth
full_df['final_depth_m'] = full_df['depth_m'].fillna(full_df['predicted_depth_m'])

# plot observed vs predicted depth over time
plt.figure(figsize=(12,5))

# observed depths
plt.plot(full_df.index, full_df['depth_m'], label='Observed depth', linewidth=1)
# filled depth record
plt.plot(full_df.index, full_df['final_depth_m'], label='Final depth record', linewidth=1)
plt.xlabel('Date')
plt.ylabel('Depth (m)')
plt.title('Observed + Predicted Depth Record')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Save and export as csv

In [ ]:
full_df.to_csv("depth_filled_record.csv")